In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.text_engine import generate_applicant_narrative, batch_generate_narratives

%matplotlib inline

Setup complete.


In [4]:
train_df = pd.read_csv('../data/application_train.csv')
prev_df = pd.read_csv('../data/previous_application.csv')

print(f"Application Train shape: {train_df.shape}")
print(f"Previous Application shape: {prev_df.shape}")

Application Train shape: (307511, 122)
Previous Application shape: (1670214, 37)


In [5]:
prev_agg = prev_df.groupby('SK_ID_CURR').agg(
    PREV_APP_COUNT=('SK_ID_PREV', 'count'),
    PREV_APPROVED_COUNT=('NAME_CONTRACT_STATUS', lambda x: (x == 'Approved').sum()),
    PREV_REFUSED_COUNT=('NAME_CONTRACT_STATUS', lambda x: (x == 'Refused').sum()),
    PRIMARY_PURPOSE=('NAME_CASH_LOAN_PURPOSE', lambda x: x.mode()[0] if not x.empty else 'general purposes')
).reset_index()

prev_agg['PRIMARY_PURPOSE'] = prev_agg['PRIMARY_PURPOSE'].replace({'XAP': 'general purposes', 'XNA': 'unspecified purposes'})

df = train_df.merge(prev_agg, on='SK_ID_CURR', how='left')
df['PREV_APP_COUNT'] = df['PREV_APP_COUNT'].fillna(0).astype(int)
df['PREV_APPROVED_COUNT'] = df['PREV_APPROVED_COUNT'].fillna(0).astype(int)
df['PREV_REFUSED_COUNT'] = df['PREV_REFUSED_COUNT'].fillna(0).astype(int)
df['PRIMARY_PURPOSE'] = df['PRIMARY_PURPOSE'].fillna('no prior loan history')

print("Merged shape:", df.shape)

Merged shape: (307511, 126)


In [8]:
df['NARRATIVE_TEXT'] = batch_generate_narratives(df)

# Print 3 sample generated narrative profiles
for idx, narrative in enumerate(df['NARRATIVE_TEXT'].head(3), 1):
    print(f"--- Sample Narrative {idx} ---")
    print(narrative)
    print()

--- Sample Narrative 1 ---
Applicant is a 25-year-old working worker in business entity type 3, residing in house / apartment. Requesting a credit amount of 406,598 USD with an annual reported income of 202,500 USD. Holds 1 previous applications (1 approved, 0 refused), primarily requested for general purposes.

--- Sample Narrative 2 ---
Applicant is a 45-year-old state servant worker in school, residing in house / apartment. Requesting a credit amount of 1,293,502 USD with an annual reported income of 270,000 USD. Holds 3 previous applications (3 approved, 0 refused), primarily requested for general purposes.

--- Sample Narrative 3 ---
Applicant is a 52-year-old working worker in government, residing in house / apartment. Requesting a credit amount of 135,000 USD with an annual reported income of 67,500 USD. Holds 1 previous applications (1 approved, 0 refused), primarily requested for general purposes.

